In [1]:
import sys
print(sys.executable)

C:\Users\yathi\anaconda3\envs\pytorch\python.exe


In [2]:
from torchvision.ops import nms
print(nms)


<function nms at 0x000002B969431120>


In [3]:
import torchvision
print(torchvision.__file__)
print(torchvision.__version__)

C:\Users\yathi\anaconda3\envs\pytorch\lib\site-packages\torchvision\__init__.py
0.26.0+cpu


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

csv_url =\
    'http://archive.ics.uci.edu/ml/machine-' \
    'learning-databases/wine-quality/winequality-red.csv'
data = pd.read_csv(csv_url, sep=';')

In [9]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [10]:
# Split the data into training and test sets. (0.75, 0.25) split.
# The predicted column is "quality" which is a scalar from [3, 9]
train_x, test_x, train_y, test_y = train_test_split(
    data.drop(['quality'], axis=1),
    data['quality']
)

In [11]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

In [14]:
import mlflow

mlflow.set_tracking_uri('http://127.0.0.1:5000')  # set to your server URI
mlflow.set_experiment("wine")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1776730590935, experiment_id='2', last_update_time=1776730590935, lifecycle_stage='active', name='wine', tags={}, trace_location=None, workspace='default'>

In [15]:
from sklearn.linear_model import ElasticNet
import mlflow.sklearn

np.random.seed(40)

def train(alpha=0.5, l1_ratio=0.5):
    with mlflow.start_run():
        lr = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
        lr.fit(train_x, train_y)
        predicted = lr.predict(test_x)
        rmse, mae, r2 = eval_metrics(test_y, predicted)

        model_name = lr.__class__.__name__
        print('{} (alpha={}, l1_ratio={}):'.format(
            model_name, alpha, l1_ratio
        ))
        print(' RMSE: %s' % rmse)
        print(' MAE: %s' % mae)
        print(' R2: %s' % r2)

        mlflow.log_params({key: value for key, value in lr.get_params().items()})
        mlflow.log_metric('rmse', rmse)
        mlflow.log_metric('r2', r2)
        mlflow.log_metric('mae', mae)
        mlflow.sklearn.log_model(lr, model_name)

In [16]:
train(0.5, 0.5)

ElasticNet (alpha=0.5, l1_ratio=0.5):
 RMSE: 0.7387688600105592
 MAE: 0.5982079755795651
 R2: 0.12939953976503238


2026/04/20 20:17:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 20:17:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run useful-ox-790 at: http://127.0.0.1:5000/#/experiments/2/runs/aba1a1de51a74466a9e4d4eba16ecbad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [17]:
train(0.1, 0.5)

ElasticNet (alpha=0.1, l1_ratio=0.5):
 RMSE: 0.6813384514527843
 MAE: 0.5386716010705862
 R2: 0.25949579609494655


2026/04/20 20:17:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 20:17:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run dazzling-seal-946 at: http://127.0.0.1:5000/#/experiments/2/runs/81b886387c03437b926a3d4b7543f61c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [18]:
train(0.8, 0.5)

2026/04/20 20:17:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


ElasticNet (alpha=0.8, l1_ratio=0.5):
 RMSE: 0.7788247585791385
 MAE: 0.6353784743414796
 R2: 0.03243259758198558


2026/04/20 20:17:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run abundant-jay-898 at: http://127.0.0.1:5000/#/experiments/2/runs/d46bf65905c3488da645698270e9c769
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [19]:
train(0.1, 0.2)

ElasticNet (alpha=0.1, l1_ratio=0.2):
 RMSE: 0.6685804988843175
 MAE: 0.5268250557693601
 R2: 0.2869678042935032


2026/04/20 20:18:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 20:18:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run exultant-dove-498 at: http://127.0.0.1:5000/#/experiments/2/runs/ba7f3833871d4bc2845c33285461341c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [20]:
train(0.1, 0.3)

ElasticNet (alpha=0.1, l1_ratio=0.3):
 RMSE: 0.6743787130475308
 MAE: 0.5319942495229919
 R2: 0.27454674013137037


2026/04/20 20:18:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 20:18:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run tasteful-kite-478 at: http://127.0.0.1:5000/#/experiments/2/runs/b556a0fce6784429a7eb80bae354e61e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [21]:
train(0.2, 0.2)

2026/04/20 20:18:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


ElasticNet (alpha=0.2, l1_ratio=0.2):
 RMSE: 0.6831621731702133
 MAE: 0.5410361607310835
 R2: 0.2555263122497231


2026/04/20 20:18:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run clean-fly-850 at: http://127.0.0.1:5000/#/experiments/2/runs/4420f7beb44b436baa0d09a1ac022980
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [22]:
train(0.1, 0.1)

ElasticNet (alpha=0.1, l1_ratio=0.1):
 RMSE: 0.6637111008574652
 MAE: 0.5224997314969303
 R2: 0.29731627787298076


2026/04/20 20:19:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 20:19:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run placid-cow-679 at: http://127.0.0.1:5000/#/experiments/2/runs/c628ca702894463f93e792cc6f9f3ac7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [23]:
train(0.05, 0.05)

ElasticNet (alpha=0.05, l1_ratio=0.05):
 RMSE: 0.6529222902533458
 MAE: 0.5126538608538342
 R2: 0.3199752478709933


2026/04/20 20:20:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 20:20:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run powerful-bug-131 at: http://127.0.0.1:5000/#/experiments/2/runs/10969be76d8a4712b5534d4dd8a1aedf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
